[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/03_cross_entropy_and_loss_functions/exercises.ipynb)

# Module 03 — Exercises: Cross-Entropy and Loss Functions

Twenty-eight solved problems in four tiers. Every problem carries a statement, a short
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it and prints the check.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): cross-entropy is $H_{\times}(p, q)$, joint
entropy is $H(X, Y)$, divergences use `\parallel`, and every numerical answer carries its unit.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps


def entropy(p, base=2.0):
    p = np.asarray(p, dtype=float)
    m = p > 0
    return float(-(p[m] * np.log(p[m])).sum() / np.log(base))


def cross_entropy(p, q, base=2.0):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = p > 0
    with np.errstate(divide="ignore"):
        return float(-(p[m] * np.log(q[m])).sum() / np.log(base))


def kl(p, q, base=2.0):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = p > 0
    with np.errstate(divide="ignore"):
        return float((p[m] * np.log(p[m] / q[m])).sum() / np.log(base))


def binary_entropy(t, base=2.0):
    t = np.asarray(t, dtype=float)
    out = np.zeros_like(t)
    m = (t > 0) & (t < 1)
    out[m] = -(t[m] * np.log(t[m]) + (1 - t[m]) * np.log(1 - t[m])) / np.log(base)
    return out


print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Cross-entropy at a perfect match

**Statement.** Let $p = q = (\tfrac12, \tfrac14, \tfrac14)$. Compute $H_{\times}(p, q)$ in bits
and say why it is not zero.

**Intuition.** A perfect model still has to pay for the source's own unpredictability.

**Solution.**

*Step 1.* At $q = p$ the definition collapses to the entropy:

$$
H_{\times}(p, p) = -\sum_k p_k \log_2 p_k = \tfrac12 (1) + \tfrac14 (2) + \tfrac14 (2) = 1.5 \text{ bits}.
$$

*Step 2.* By Theorem 4.1 the excess over $H(p)$ is $D_{\mathrm{KL}}(p \parallel q)$, which is
what vanishes at a perfect match — not the loss itself.

$$
\boxed{H_{\times}(p, p) = H(p) = 1.5 \text{ bits}}
$$

**Key takeaway.** The reachable minimum of a cross-entropy loss is $H(p)$, never zero; compare
losses to the entropy floor.

In [2]:
p = np.array([0.5, 0.25, 0.25])
print(f"H_x(p, p) = {cross_entropy(p, p):.4f} bits   H(p) = {entropy(p):.4f} bits")
print(f"D_KL(p || p) = {kl(p, p):.3e} bits")
assert abs(cross_entropy(p, p) - 1.5) < 8 * EPS
assert abs(kl(p, p)) < 8 * EPS

H_x(p, p) = 1.5000 bits   H(p) = 1.5000 bits
D_KL(p || p) = 0.000e+00 bits


### Problem L0.2 — One-hot cross-entropy

**Statement.** For a one-hot target $y = (0, 1, 0)$ and prediction $q = (0.2, 0.7, 0.1)$,
compute the loss in nats.

**Intuition.** A one-hot target puts all the weight on a single term of the sum.

**Solution.**

*Step 1.* Only the true-class term survives:

$$
H_{\times}(y, q) = -\sum_k y_k \ln q_k = -\ln q_2 = -\ln 0.7 = 0.35667 \text{ nats}.
$$

$$
\boxed{\mathcal{L} = -\ln 0.7 = 0.35667 \text{ nats}}
$$

**Key takeaway.** With hard labels the loss is exactly the negative log-probability assigned to
the correct class.

In [3]:
y, q = np.array([0.0, 1.0, 0.0]), np.array([0.2, 0.7, 0.1])
print(f"loss = {cross_entropy(y, q, np.e):.5f} nats = {-np.log(0.7):.5f} nats")
assert abs(cross_entropy(y, q, np.e) + np.log(0.7)) < 8 * EPS

loss = 0.35667 nats = 0.35667 nats


### Problem L0.3 — The infinite penalty

**Statement.** A model assigns $q(x_0) = 0$ to an outcome with $p(x_0) = 0.01$. What is
$H_{\times}(p, q)$, and what keeps deployed systems away from this?

**Intuition.** A declared-impossible event that happens costs an infinite number of bits.

**Solution.**

*Step 1.* The term $-p(x_0)\log q(x_0) = -0.01 \log 0 = +\infty$ dominates every finite term,
so $H_{\times}(p, q) = +\infty$. This is the failure of absolute continuity in Theorem 4.1.

*Step 2.* Practice keeps $q \gt 0$ everywhere: additive (Laplace) smoothing
$q_k \leftarrow (n_k + \alpha)/(N + \alpha K)$, clipping, or mixing with a uniform floor.

$$
\boxed{H_{\times}(p, q) = +\infty; \text{ smoothing keeps } q \gt 0 \text{ everywhere}}
$$

**Key takeaway.** Never assign probability exactly zero to anything that can happen — under log
loss the penalty is unbounded.

In [4]:
p3 = np.array([0.99, 0.01])
q3 = np.array([1.0, 0.0])
print("H_x with q(x0) = 0 :", cross_entropy(p3, q3), "bits")
for alpha in [1e-2, 1e-4, 1e-6]:
    q_s = np.array([1.0 - alpha, alpha])
    print(f"  Laplace floor alpha = {alpha:.0e}:  H_x = {cross_entropy(p3, q_s):.4f} bits")
assert np.isinf(cross_entropy(p3, q3))

H_x with q(x0) = 0 : inf bits
  Laplace floor alpha = 1e-02:  H_x = 0.0808 bits
  Laplace floor alpha = 1e-04:  H_x = 0.1330 bits
  Laplace floor alpha = 1e-06:  H_x = 0.1993 bits


### Problem L0.4 — Cross-entropy is asymmetric

**Statement.** Let $p = (0.9, 0.1)$ and $q = (0.5, 0.5)$. Compute $H_{\times}(p, q)$ and
$H_{\times}(q, p)$ in bits.

**Intuition.** The first argument owns the frequencies, the second owns the code; swapping them
changes the bill.

**Solution.**

*Step 1.* $H_{\times}(p, q) = -0.9\log_2 0.5 - 0.1\log_2 0.5 = 0.9 + 0.1 = 1$ bit.

*Step 2.* $H_{\times}(q, p) = -0.5\log_2 0.9 - 0.5\log_2 0.1 = 0.5(0.15200) + 0.5(3.32193) = 1.73697$ bits.

$$
\boxed{H_{\times}(p, q) = 1 \text{ bit} \neq H_{\times}(q, p) = 1.73697 \text{ bits}}
$$

**Key takeaway.** Coding a skewed source with a uniform code wastes little; coding a uniform
source with a skewed code wastes a lot, because a long codeword goes to a frequent outcome.

In [5]:
p4, q4 = np.array([0.9, 0.1]), np.array([0.5, 0.5])
print(f"H_x(p, q) = {cross_entropy(p4, q4):.5f} bits")
print(f"H_x(q, p) = {cross_entropy(q4, p4):.5f} bits")
assert abs(cross_entropy(p4, q4) - 1.0) < 8 * EPS
assert abs(cross_entropy(q4, p4) - 1.73697) < 5e-6

H_x(p, q) = 1.00000 bits
H_x(q, p) = 1.73697 bits


### Problem L0.5 — The loss of an untrained classifier

**Statement.** A freshly initialized classifier over $K = 1000$ classes outputs equal logits.
What loss does it report, in nats and in bits?

**Intuition.** Equal logits mean a uniform prediction, and a uniform prediction pays $\log K$
whatever the labels are.

**Solution.**

*Step 1.* Equal logits give $q_k = 1/K$ for every $k$, so for any target on the simplex

$$
H_{\times}(y, q) = -\sum_k y_k \log \frac{1}{K} = \log K .
$$

*Step 2.* $\ln 1000 = 6.90776$ nats, and dividing by $\ln 2$ gives $9.96578$ bits.

$$
\boxed{\mathcal{L}_{\text{init}} = \ln 1000 = 6.90776 \text{ nats} = 9.96578 \text{ bits}}
$$

**Key takeaway.** The first loss value a training run prints should be $\log K$; anything else
means the head, the targets or the reduction is wrong.

In [6]:
K = 1000
print(f"log K = {np.log(K):.5f} nats = {np.log2(K):.5f} bits")
z = np.zeros(K)
q = np.exp(z) / np.exp(z).sum()
y = np.zeros(K)
y[17] = 1.0
print(f"empirical loss on a one-hot target: {cross_entropy(y, q, np.e):.5f} nats")
assert abs(cross_entropy(y, q, np.e) - np.log(K)) < 1e-12

log K = 6.90776 nats = 9.96578 bits
empirical loss on a one-hot target: 6.90776 nats


### Problem L0.6 — Softmax gradients sum to zero

**Statement.** Show that the softmax cross-entropy gradient of Theorem 4.5 always has entries
summing to zero.

**Intuition.** Adding a constant to every logit changes nothing, so the loss cannot vary along
that direction.

**Solution.**

*Step 1.* By Theorem 4.5, $\partial \mathcal{L}/\partial z_m = q_m - y_m$.

*Step 2.* Sum over $m$: $\sum_m q_m = 1$ and $\sum_m y_m = 1$, so the total is $0$.

$$
\boxed{\sum_m \frac{\partial \mathcal{L}}{\partial z_m} = 1 - 1 = 0}
$$

**Key takeaway.** Softmax cross-entropy gradients live on the zero-sum hyperplane, which is the
same statement as the null eigenvector $\mathbf{1}$ of the Hessian in Example 6.4.

In [7]:
z6 = np.array([1.3, -0.4, 2.2, 0.0])
q6 = np.exp(z6) / np.exp(z6).sum()
y6 = np.array([0.0, 1.0, 0.0, 0.0])
print("gradient:", q6 - y6, "  sum =", f"{(q6 - y6).sum():.3e}")
shifted = np.exp(z6 + 7.5) / np.exp(z6 + 7.5).sum()
print("loss at z and at z + 7.5:",
      f"{cross_entropy(y6, q6, np.e):.10f}", f"{cross_entropy(y6, shifted, np.e):.10f}")
assert abs((q6 - y6).sum()) < 8 * EPS
assert abs(cross_entropy(y6, q6, np.e) - cross_entropy(y6, shifted, np.e)) < 1e-12

gradient: [ 0.2554 -0.9533  0.6283  0.0696]   sum = -2.776e-17
loss at z and at z + 7.5: 3.0647689499 3.0647689499


## L1 — Foundations

### Problem L1.1 — Verify the decomposition

**Statement.** For $p = (0.8, 0.2)$ and $q = (0.6, 0.4)$ compute $H(p)$, $H_{\times}(p, q)$ and
$D_{\mathrm{KL}}(p \parallel q)$ in bits, and verify Theorem 4.1.

**Intuition.** The identity is term-by-term algebra, not an approximation.

**Solution.**

*Step 1 — entropy.*

$$
H(p) = -0.8\log_2 0.8 - 0.2\log_2 0.2 = 0.8(0.32193) + 0.2(2.32193) = 0.72193 \text{ bits}.
$$

*Step 2 — cross-entropy.*

$$
H_{\times}(p, q) = -0.8\log_2 0.6 - 0.2\log_2 0.4 = 0.8(0.73697) + 0.2(1.32193) = 0.85396 \text{ bits}.
$$

*Step 3 — divergence.*

$$
D_{\mathrm{KL}}(p \parallel q) = 0.8\log_2\tfrac{0.8}{0.6} + 0.2\log_2\tfrac{0.2}{0.4}
= 0.8(0.41504) - 0.2 = 0.13203 \text{ bits}.
$$

*Step 4 — check.* $0.72193 + 0.13203 = 0.85396$.

$$
\boxed{H_{\times}(p, q) = 0.85396 = 0.72193 + 0.13203 \text{ bits}}
$$

**Key takeaway.** The second KL term here is negative and the identity still holds: only the
$p$-weighted total is forced non-negative.

In [8]:
p11, q11 = np.array([0.8, 0.2]), np.array([0.6, 0.4])
H11, CE11, KL11 = entropy(p11), cross_entropy(p11, q11), kl(p11, q11)
print(f"H(p) = {H11:.5f}   H_x(p,q) = {CE11:.5f}   KL = {KL11:.5f} bits")
print("per-term KL contributions:", 0.8 * np.log2(0.8 / 0.6), 0.2 * np.log2(0.2 / 0.4))
print(f"residual = {abs(CE11 - H11 - KL11):.3e}")
assert abs(CE11 - H11 - KL11) < 8 * EPS
assert abs(H11 - 0.72193) < 5e-6 and abs(KL11 - 0.13203) < 5e-6

H(p) = 0.72193   H_x(p,q) = 0.85396   KL = 0.13203 bits
per-term KL contributions: 0.3320299994230752 -0.2
residual = 5.551e-17


### Problem L1.2 — The optimal constant predictor

**Statement.** A binary dataset is $70$ percent positive. Show that the constant prediction
minimizing average BCE is $\hat{p} = 0.7$, and compute the resulting loss in nats.

**Intuition.** Log loss is strictly proper, so the best constant report is the true base rate.

**Solution.**

*Step 1.* The average loss of a constant $\hat{p}$ is

$$
\mathcal{L}(\hat{p}) = -0.7 \ln \hat{p} - 0.3 \ln (1 - \hat{p}) .
$$

*Step 2.* Differentiate and set to zero:

$$
\mathcal{L}'(\hat{p}) = -\frac{0.7}{\hat{p}} + \frac{0.3}{1 - \hat{p}} = 0
\;\Longrightarrow\; 0.7(1 - \hat{p}) = 0.3 \hat{p} \;\Longrightarrow\; \hat{p} = 0.7 .
$$

*Step 3.* $\mathcal{L}''(\hat p) = 0.7/\hat p^2 + 0.3/(1-\hat p)^2 \gt 0$, so this is the global
minimum.

*Step 4.* The minimum value is the base-rate entropy:

$$
\mathcal{L}(0.7) = -0.7\ln 0.7 - 0.3\ln 0.3 = H_b(0.3) = 0.61086 \text{ nats}.
$$

$$
\boxed{\hat{p}^{\star} = 0.7, \qquad \mathcal{L}^{\star} = H_b(0.3) = 0.61086 \text{ nats}}
$$

**Key takeaway.** A model that does not beat $H_b(\text{base rate})$ is not using its features
at all — this is the honest baseline for any binary task.

In [9]:
grid = np.linspace(1e-4, 1 - 1e-4, 200001)
loss = -0.7 * np.log(grid) - 0.3 * np.log(1 - grid)
print(f"argmin over the grid = {grid[loss.argmin()]:.5f}   (predicted 0.7)")
print(f"minimum loss         = {loss.min():.5f} nats")
print(f"H_b(0.3)             = {float(binary_entropy(0.3, np.e)):.5f} nats")
assert abs(grid[loss.argmin()] - 0.7) < 1e-4
assert abs(loss.min() - float(binary_entropy(0.3, np.e))) < 1e-8

argmin over the grid = 0.70000   (predicted 0.7)
minimum loss         = 0.61086 nats
H_b(0.3)             = 0.61086 nats


### Problem L1.3 — A softmax gradient by hand

**Statement.** Logits $z = (2, 0, -1)$ with true class $1$, so $y = (1, 0, 0)$. Compute $q$, the
loss in nats, and $\nabla_z \mathcal{L}$.

**Intuition.** Exponentiate, normalize, subtract the target.

**Solution.**

*Step 1.* $(e^2, e^0, e^{-1}) = (7.38906, 1, 0.36788)$ and $Z = 8.75694$.

*Step 2.* $q = (0.84380, 0.11419, 0.04201)$.

*Step 3.* $\mathcal{L} = -\ln q_1 = 0.16985$ nats.

*Step 4.* By Theorem 4.5, $\nabla_z \mathcal{L} = q - y = (-0.15620, 0.11419, 0.04201)$, whose
entries sum to zero.

$$
\boxed{\mathcal{L} = 0.16985 \text{ nats}, \qquad \nabla_z \mathcal{L} = (-0.15620,\, 0.11419,\, 0.04201)}
$$

**Key takeaway.** The true class is pushed up by its probability deficit and every other class
is pushed down by its own probability — no derivative of an exponential ever appears.

In [10]:
z13 = np.array([2.0, 0.0, -1.0])
q13 = np.exp(z13) / np.exp(z13).sum()
y13 = np.array([1.0, 0.0, 0.0])
print("q            =", q13)
print(f"loss         = {-np.log(q13[0]):.5f} nats")
print("gradient q-y =", q13 - y13)
g_fd = np.array([(cross_entropy(y13, np.exp(z13 + h) / np.exp(z13 + h).sum(), np.e)
                  - cross_entropy(y13, np.exp(z13 - h) / np.exp(z13 - h).sum(), np.e)) / 2e-6
                 for h in np.eye(3) * 1e-6])
print("central differences:", g_fd)
assert np.abs(g_fd - (q13 - y13)).max() < 1e-8

q            = [0.8438 0.1142 0.042 ]
loss         = 0.16985 nats
gradient q-y = [-0.1562  0.1142  0.042 ]
central differences: [-0.1562  0.1142  0.042 ]


### Problem L1.4 — Affine in $p$, convex in $q$

**Statement.** Prove that $p \mapsto H_{\times}(p, q)$ is affine and that
$q \mapsto H_{\times}(p, q)$ is convex on the simplex. State precisely where the convexity is
strict.

**Intuition.** The first argument only supplies weights; the second sits inside a logarithm.

**Solution.**

*Step 1 — affine in $p$.* $H_{\times}(p, q) = \sum_x p(x)\left(-\log q(x)\right)$ is a linear
functional of $p$ for fixed $q$. This is what licensed the label-smoothing split of Proof 5.8.

*Step 2 — convex in $q$.* Each map $t \mapsto -\log t$ has second derivative $1/t^2 \gt 0$ and
is therefore convex, and $H_{\times}(p, \cdot)$ is a non-negative combination of such maps.

*Step 3 — where strictness holds.* The Hessian in $q$ is
$\operatorname{diag}\left(p(x)/q(x)^2\right)$, which is positive definite only in the
coordinates with $p(x) \gt 0$. In the coordinates with $p(x) = 0$ the function is **constant**,
not strictly convex.

*Step 4 — why the minimizer is still unique.* On the simplex, $q = p$ is forced to put zero
mass outside $\operatorname{supp}(p)$ by normalization, since matching $p$ on its support
already uses all the mass. Uniqueness therefore comes from Theorem 4.1, not from strict
convexity in every coordinate.

$$
\boxed{H_{\times} \text{ is affine in } p, \text{ convex in } q, \text{ strictly convex only in the coordinates } \{q(x) : p(x) \gt 0\}}
$$

**Key takeaway.** Convexity lives on the model's side of the loss; the strictness is restricted
to the support of the data, which is exactly why unseen classes are unidentifiable.

In [11]:
p14 = np.array([0.6, 0.4, 0.0])
q_a, q_b = np.array([0.5, 0.3, 0.2]), np.array([0.3, 0.5, 0.2])
for lam in [0.25, 0.5, 0.75]:
    mid = lam * q_a + (1 - lam) * q_b
    lhs = cross_entropy(p14, mid)
    rhs = lam * cross_entropy(p14, q_a) + (1 - lam) * cross_entropy(p14, q_b)
    print(f"lambda = {lam}:  H_x(mid) = {lhs:.6f}  <=  {rhs:.6f} = convex combination")
    assert lhs <= rhs + 8 * EPS
print("\nflat direction: p(x3) = 0, so H_x does not move when q3 changes at fixed q1, q2")
for q3 in [0.0, 0.1, 0.3]:
    q_t = np.array([0.6, 0.4, q3])
    print(f"  q3 = {q3}:  H_x = {cross_entropy(p14, q_t):.6f} bits (unnormalized q)")
assert abs(cross_entropy(p14, np.array([0.6, 0.4, 0.3]))
           - cross_entropy(p14, np.array([0.6, 0.4, 0.0]))) < 8 * EPS

lambda = 0.25:  H_x(mid) = 1.369545  <=  1.405331 = convex combination
lambda = 0.5:  H_x(mid) = 1.321928  <=  1.368483 = convex combination
lambda = 0.75:  H_x(mid) = 1.297031  <=  1.331635 = convex combination

flat direction: p(x3) = 0, so H_x does not move when q3 changes at fixed q1, q2
  q3 = 0.0:  H_x = 0.970951 bits (unnormalized q)
  q3 = 0.1:  H_x = 0.970951 bits (unnormalized q)
  q3 = 0.3:  H_x = 0.970951 bits (unnormalized q)


### Problem L1.5 — Gaussian cross-entropy is mean squared error

**Statement.** Show that if the model is $q(y \mid x) = \mathcal{N}(y; \mu_\theta(x), \sigma^2)$
with $\sigma$ fixed, minimizing the expected cross-entropy is minimizing MSE.

**Intuition.** The log of a Gaussian density is a quadratic plus a constant.

**Solution.**

*Step 1.* The pointwise loss is

$$
-\ln q(y \mid x) = \frac{1}{2}\ln\left(2\pi\sigma^2\right) + \frac{\left(y - \mu_\theta(x)\right)^2}{2\sigma^2}.
$$

*Step 2.* Averaging over the data, the first term does not depend on $\theta$, so

$$
\operatorname*{arg\,min}_\theta \mathbb{E}\left[-\ln q(Y \mid X)\right]
= \operatorname*{arg\,min}_\theta \mathbb{E}\left[\left(Y - \mu_\theta(X)\right)^2\right].
$$

*Step 3.* If $\sigma$ is *learned* the equivalence breaks: the $\tfrac12\ln(2\pi\sigma^2)$ term
becomes $\theta$-dependent and the objective becomes a heteroscedastic likelihood.

$$
\boxed{\text{Gaussian NLL with fixed } \sigma \;\equiv\; \text{MSE, up to an additive constant and the factor } 1/(2\sigma^2)}
$$

**Key takeaway.** MSE is not an alternative to cross-entropy; it *is* cross-entropy under a
homoscedastic Gaussian observation model. Every loss encodes a distributional assumption.

In [12]:
sigma = 0.7
mu_grid = np.linspace(-2.0, 4.0, 6001)
y_obs = np.array([1.2, 0.4, 2.5, 1.9, 0.8])
nll = np.array([0.5 * np.log(2 * np.pi * sigma ** 2) * y_obs.size
                + ((y_obs - m) ** 2).sum() / (2 * sigma ** 2) for m in mu_grid])
mse = np.array([((y_obs - m) ** 2).mean() for m in mu_grid])
print(f"argmin NLL = {mu_grid[nll.argmin()]:.4f}   argmin MSE = {mu_grid[mse.argmin()]:.4f}"
      f"   sample mean = {y_obs.mean():.4f}")
assert abs(mu_grid[nll.argmin()] - mu_grid[mse.argmin()]) < 1e-9
assert abs(mu_grid[nll.argmin()] - y_obs.mean()) < 1e-3

argmin NLL = 1.3600   argmin MSE = 1.3600   sample mean = 1.3600


### Problem L1.6 — The stable BCE-with-logits formula

**Statement.** Derive
$\mathcal{L}(z, y) = \max(z, 0) - zy + \ln\left(1 + e^{-\lvert z \rvert}\right)$ and explain why
the naive form fails at $z = -800$, $y = 1$.

**Intuition.** Never exponentiate a large logit; rearrange so the exponent is always negative.

**Solution.**

*Step 1.* With $\hat{p} = \sigma(z)$, $-\ln \hat{p} = \ln(1 + e^{-z})$ and
$-\ln(1 - \hat{p}) = \ln(1 + e^{z})$, so

$$
\mathcal{L} = y \ln\left(1 + e^{-z}\right) + (1-y)\ln\left(1 + e^{z}\right).
$$

*Step 2.* Using $\ln(1 + e^{z}) = z + \ln(1 + e^{-z})$ this becomes
$\mathcal{L} = \ln(1 + e^{-z}) + (1 - y) z$.

*Step 3.* Applying the same identity again when $z \lt 0$ moves the sign of the exponent, and
the two branches merge into

$$
\mathcal{L}(z, y) = \max(z, 0) - zy + \ln\left(1 + e^{-\lvert z \rvert}\right),
$$

whose exponent is never positive. Check: at $z \ge 0$ it reads $z - zy + \ln(1+e^{-z})$; at
$z \lt 0$ it reads $-zy + \ln(1+e^{z})$.

*Step 4 — the failure.* At $z = -800$ the value $\sigma(z)$ underflows to exactly $0$, so
$-\ln \hat{p} = +\infty$. The stable form returns $0 + 800 + \ln(1 + e^{-800}) = 800$, the
correct finite loss.

$$
\boxed{\mathcal{L}(z, y) = \max(z, 0) - zy + \ln\left(1 + e^{-\lvert z \rvert}\right)}
$$

**Key takeaway.** Fusing the sigmoid into the loss removes both overflow and the logarithm of a
rounded-to-zero probability — the reason `BCEWithLogitsLoss` takes logits, not probabilities.

In [13]:
def bce_stable(z, y):
    return np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))


zs = np.array([-800.0, -50.0, -1.0, 0.0, 1.0, 50.0, 800.0])
ys = np.ones_like(zs)
with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
    sig = 1.0 / (1.0 + np.exp(-zs))
    naive = -ys * np.log(sig) - (1 - ys) * np.log(1 - sig)
print("z      :", zs)
print("naive  :", naive)
print("stable :", bce_stable(zs, ys))
mid = np.array([-3.0, 0.5, 2.0])
print("\nagreement in the safe range:",
      np.abs(bce_stable(mid, np.ones(3)) + np.log(1 / (1 + np.exp(-mid)))).max())
assert np.isfinite(bce_stable(zs, ys)).all()
assert abs(bce_stable(np.array([-800.0]), np.array([1.0]))[0] - 800.0) < 1e-9
assert not np.isfinite(naive).all()

z      : [-800.  -50.   -1.    0.    1.   50.  800.]
naive  : [    inf 50.      1.3133  0.6931  0.3133     nan     nan]
stable : [800.      50.       1.3133   0.6931   0.3133   0.       0.    ]

agreement in the safe range: 1.3877787807814457e-16


### Problem L1.7 — The loss floor of a noisy channel

**Statement.** A binary label $Y$ is produced from a binary feature $X$ by the joint table
$p(x, y)$ with entries $p(0,0) = p(1,1) = 0.4$ and $p(0,1) = p(1,0) = 0.1$. What is the smallest
expected cross-entropy any predictor of $Y$ from $X$ can achieve, in bits?

**Intuition.** The best predictor reports the true conditional distribution, and pays its
entropy.

**Solution.**

*Step 1 — marginals.* $p(x) = (0.5, 0.5)$ and $p(y) = (0.5, 0.5)$, so a predictor ignoring $X$
pays $H(Y) = 1$ bit.

*Step 2 — conditionals.* $p(y \mid x=0) = (0.8, 0.2)$ and $p(y \mid x=1) = (0.2, 0.8)$.

*Step 3 — the floor.* For each $x$, Theorem 4.1 says the reported $q(\cdot \mid x)$ minimizing
expected loss is $p(\cdot \mid x)$, with value $H(Y \mid X = x) = H_b(0.2)$. Averaging over $x$,

$$
H(Y \mid X) = H_b(0.2) = 0.72193 \text{ bits}.
$$

*Step 4 — consistency check.* $H(X, Y) = 1.72193$ and $H(X) = 1$, so
$H(Y \mid X) = H(X,Y) - H(X) = 0.72193$ bits, matching the chain rule of
[Module 02](../02_joint_and_conditional_entropy/first_principles.ipynb).

$$
\boxed{\min_q \mathbb{E}\left[H_{\times}\right] = H(Y \mid X) = H_b(0.2) = 0.72193 \text{ bits}}
$$

**Key takeaway.** Every predictive task has a floor set by the conditional entropy; the gap
between a model's loss and that floor is the only part worth optimizing.

In [14]:
J = np.array([[0.4, 0.1], [0.1, 0.4]])
px, py = J.sum(axis=1), J.sum(axis=0)
cond = J / px[:, None]
HYgX = float(sum(px[i] * entropy(cond[i]) for i in range(2)))
print("p(x) =", px, " p(y) =", py)
print("p(y|x) =\n", cond)
print(f"H(Y)      = {entropy(py):.5f} bits")
print(f"H(Y | X)  = {HYgX:.5f} bits   H_b(0.2) = {float(binary_entropy(0.2)):.5f} bits")
print(f"H(X,Y) - H(X) = {entropy(J.ravel()) - entropy(px):.5f} bits")
worse = np.array([0.7, 0.3])
print("loss of a predictor reporting (0.7, 0.3) given x = 0:",
      f"{cross_entropy(cond[0], worse):.5f} bits")
assert abs(HYgX - float(binary_entropy(0.2))) < 1e-12
assert abs(HYgX - (entropy(J.ravel()) - entropy(px))) < 1e-12
assert cross_entropy(cond[0], worse) > HYgX

p(x) = [0.5 0.5]  p(y) = [0.5 0.5]
p(y|x) =
 [[0.8 0.2]
 [0.2 0.8]]
H(Y)      = 1.00000 bits
H(Y | X)  = 0.72193 bits   H_b(0.2) = 0.72193 bits
H(X,Y) - H(X) = 0.72193 bits
loss of a predictor reporting (0.7, 0.3) given x = 0: 0.75905 bits


### Problem L1.8 — The Hessian of the softmax loss

**Statement.** Show that $\nabla_z^2 \mathcal{L} = \operatorname{diag}(q) - q q^{\top}$ is
positive semidefinite with $\mathbf{1}$ in its null space, and compute its eigenvalues for
$z = (2, 0, -1)$.

**Intuition.** The Hessian is a covariance matrix under $q$, and a covariance is never negative.

**Solution.**

*Step 1.* By Proof 5.5, $\partial q_m / \partial z_n = q_m(\delta_{mn} - q_n)$, so the Hessian
of $\mathcal{L}$ is $\operatorname{diag}(q) - qq^{\top}$.

*Step 2 — semidefiniteness.* For any $v$,

$$
v^{\top}\left(\operatorname{diag}(q) - qq^{\top}\right) v = \sum_k q_k v_k^2 - \left(\sum_k q_k v_k\right)^2 = \operatorname{Var}_{k \sim q}(v_k) \ge 0 .
$$

*Step 3 — the null direction.* The variance is zero exactly for constant $v$, so the null space
is spanned by $\mathbf{1}$ and the loss is flat along a uniform logit shift.

*Step 4 — the numbers.* With $q = (0.84380, 0.11419, 0.04201)$ the eigenvalues are

$$
0, \qquad 0.05588, \qquad 0.21733 .
$$

$$
\boxed{\nabla^2_z \mathcal{L} = \operatorname{diag}(q) - qq^{\top} \succeq 0, \quad \text{null space} = \operatorname{span}(\mathbf{1})}
$$

**Key takeaway.** The per-example loss is convex in the logits with exactly one flat direction,
so second-order methods on the last layer are well-behaved once that direction is fixed.

In [15]:
z18 = np.array([2.0, 0.0, -1.0])
q18 = np.exp(z18) / np.exp(z18).sum()
Hs = np.diag(q18) - np.outer(q18, q18)
evals = np.linalg.eigvalsh(Hs)
print("q          =", q18)
print("Hessian    =\n", Hs)
print("eigenvalues:", evals)
print("H @ 1      =", Hs @ np.ones(3))
v = rng.normal(size=3)
print(f"v^T H v = {v @ Hs @ v:.6f}   Var_q(v) = {float((q18 * v ** 2).sum() - (q18 @ v) ** 2):.6f}")
assert evals.min() > -8 * EPS
assert np.abs(Hs @ np.ones(3)).max() < 8 * EPS
assert np.allclose(evals[1:], [0.05588, 0.21733], atol=5e-6)

q          = [0.8438 0.1142 0.042 ]
Hessian    =
 [[ 0.1318 -0.0964 -0.0354]
 [-0.0964  0.1012 -0.0048]
 [-0.0354 -0.0048  0.0402]]
eigenvalues: [-0.      0.0559  0.2173]
H @ 1      = [0. 0. 0.]
v^T H v = 0.018659   Var_q(v) = 0.018659


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Perplexity improvement in bits

**Statement.** Model A reports a per-token cross-entropy of $3.466$ nats, model B reports
$3.120$ nats. Compute both perplexities, the saving in bits per token, and the compression
consequence over $10^9$ tokens.

**Intuition.** Perplexity is an exponentiated loss and bits per token is a rescaled one; the
same number in three costumes.

**Solution.**

*Step 1 — perplexities.* $e^{3.466} = 32.01$ and $e^{3.120} = 22.65$.

*Step 2 — bits.* $3.466/\ln 2 = 5.00038$ and $3.120/\ln 2 = 4.50121$ bits per token.

*Step 3 — saving.* The difference is $0.49917$ bits per token.

*Step 4 — compression.* By Theorem 4.3 an arithmetic coder driven by model B writes within one
bit of that per token, so $10^9$ tokens shrink by
$0.49917 \times 10^9 / 8 = 6.24 \times 10^7$ bytes, about $62.4$ MB.

$$
\boxed{\mathrm{PPL}: 32.01 \to 22.65, \quad \text{saving } 0.49917 \text{ bits/token} \approx 62.4 \text{ MB per } 10^9 \text{ tokens}}
$$

**Key takeaway.** Language-model progress is compression progress, literally: every $\ln 2$
nats of cross-entropy removed halves the perplexity and saves one bit per token.

In [16]:
ce_a, ce_b = 3.466, 3.120
bits_a, bits_b = ce_a / np.log(2), ce_b / np.log(2)
print(f"PPL A = {np.exp(ce_a):.2f}   PPL B = {np.exp(ce_b):.2f}")
print(f"bits/token A = {bits_a:.5f}   B = {bits_b:.5f}   saving = {bits_a - bits_b:.5f}")
print(f"bytes saved over 1e9 tokens = {(bits_a - bits_b) * 1e9 / 8:.4e}"
      f"  = {(bits_a - bits_b) * 1e9 / 8 / 1e6:.1f} MB")
print(f"halving check: exp(ln 2) = {np.exp(np.log(2)):.4f}")
assert abs(np.exp(ce_a) - 32.01) < 0.01 and abs(np.exp(ce_b) - 22.65) < 0.01
assert abs((bits_a - bits_b) - 0.49917) < 5e-6

PPL A = 32.01   PPL B = 22.65
bits/token A = 5.00038   B = 4.50121   saving = 0.49917
bytes saved over 1e9 tokens = 6.2397e+07  = 62.4 MB
halving check: exp(ln 2) = 2.0000


### Problem L2.2 — Label smoothing targets and optimal confidence

**Statement.** With $K = 10$ classes and $\epsilon = 0.1$, write the smoothed target for true
class $3$ and compute the optimal confidence and the optimal logit gap.

**Intuition.** Theorem 4.8 says the optimum is the smoothed target itself.

**Solution.**

*Step 1.* The smoothed target is $\tilde{y}_3 = 1 - \epsilon + \epsilon/K = 0.91$ and
$\tilde{y}_k = \epsilon/K = 0.01$ for $k \neq 3$.

*Step 2.* By Theorem 4.8 the loss-minimizing prediction is $q^{\star} = \tilde{y}$, so optimal
confidence is $0.91$.

*Step 3.* Inverting the softmax,

$$
z_{\text{true}} - z_{\text{other}} = \ln \frac{0.91}{0.01} = \ln 91 = 4.51086 .
$$

$$
\boxed{q^{\star}_{\text{true}} = 0.91, \qquad \text{logit gap} = \ln 91 = 4.51086}
$$

**Key takeaway.** Smoothing converts "push the logit to infinity" into a finite target, which is
why it improves calibration and gradient conditioning at once.

In [17]:
K22, eps22 = 10, 0.1
tgt = np.full(K22, eps22 / K22)
tgt[2] += 1 - eps22
print("smoothed target:", tgt, " sums to", tgt.sum())
print(f"optimal confidence = {tgt[2]:.4f}   logit gap = {np.log(tgt[2] / tgt[0]):.5f} = ln 91")
z_opt = np.zeros(K22)
z_opt[2] = np.log(91)
q_opt = np.exp(z_opt) / np.exp(z_opt).sum()
print("softmax of those logits:", q_opt)
assert abs(tgt[2] - 0.91) < 8 * EPS
assert abs(np.log(tgt[2] / tgt[0]) - np.log(91)) < 1e-12
assert np.allclose(q_opt, tgt)

smoothed target: [0.01 0.01 0.91 0.01 0.01 0.01 0.01 0.01 0.01 0.01]  sums to 1.0
optimal confidence = 0.9100   logit gap = 4.51086 = ln 91
softmax of those logits: [0.01 0.01 0.91 0.01 0.01 0.01 0.01 0.01 0.01 0.01]


### Problem L2.3 — Focal loss on easy and hard examples

**Statement.** Focal loss is $\mathrm{FL}(q_t) = -(1 - q_t)^{\gamma} \ln q_t$, where $q_t$ is
the probability given to the true class. For $\gamma = 2$, compare it with cross-entropy at
$q_t = 0.9$ and $q_t = 0.1$.

**Intuition.** The modulating factor is small exactly when the example is already easy.

**Solution.**

*Step 1 — cross-entropy.* $-\ln 0.9 = 0.10536$ and $-\ln 0.1 = 2.30259$ nats.

*Step 2 — modulating factors.* $(1 - 0.9)^2 = 0.01$ and $(1 - 0.1)^2 = 0.81$.

*Step 3 — focal losses.* $0.01 \times 0.10536 = 0.00105$ and $0.81 \times 2.30259 = 1.86509$.

*Step 4 — relative emphasis.* The hard-to-easy ratio moves from
$2.30259/0.10536 = 21.85$ under cross-entropy to $1.86509/0.00105 = 1770.2$ under focal loss, a
factor of $81 = (0.81/0.01)$.

$$
\boxed{\text{easy } 0.10536 \to 0.00105, \quad \text{hard } 2.30259 \to 1.86509, \quad \text{ratio } 21.85 \to 1770.2}
$$

**Key takeaway.** Focal loss silences the flood of well-classified background examples so that
rare hard positives dominate the gradient — the trick behind one-stage detectors.

In [18]:
gamma = 2.0
for qt in [0.9, 0.1]:
    ce = -np.log(qt)
    fl = (1 - qt) ** gamma * ce
    print(f"q_t = {qt}:  CE = {ce:.5f}   factor = {(1 - qt) ** gamma:.4f}   FL = {fl:.5f}")
ce_easy, ce_hard = -np.log(0.9), -np.log(0.1)
fl_easy, fl_hard = 0.01 * ce_easy, 0.81 * ce_hard
print(f"ratio under CE = {ce_hard / ce_easy:.2f}   under FL = {fl_hard / fl_easy:.1f}"
      f"   quotient = {(fl_hard / fl_easy) / (ce_hard / ce_easy):.1f}")
assert abs(fl_hard - 1.86509) < 1e-5 and abs(fl_easy - 0.00105) < 1e-5
assert abs((fl_hard / fl_easy) / (ce_hard / ce_easy) - 81.0) < 1e-9

q_t = 0.9:  CE = 0.10536   factor = 0.0100   FL = 0.00105
q_t = 0.1:  CE = 2.30259   factor = 0.8100   FL = 1.86509
ratio under CE = 21.85   under FL = 1770.2   quotient = 81.0


### Problem L2.4 — Distillation gradients and the $T^2$ rescaling

**Statement.** A student minimizes cross-entropy against a teacher's tempered softmax
$p^{(T)} \propto e^{u_k/T}$ using its own $q^{(T)} = \operatorname{softmax}(z/T)$. Show the
gradient in the student's logits is $\tfrac{1}{T}\left(q^{(T)} - p^{(T)}\right)$ and explain the
conventional $T^2$ factor.

**Intuition.** Temperature divides the logits, so the chain rule divides the gradient.

**Solution.**

*Step 1.* Write $\tilde{z} = z/T$. The targets $p^{(T)}$ sum to $1$, so Theorem 4.5 applies to
$\tilde{z}$:

$$
\frac{\partial \mathcal{L}}{\partial \tilde{z}_k} = q^{(T)}_k - p^{(T)}_k .
$$

*Step 2.* By the chain rule $\partial \tilde{z}_k / \partial z_k = 1/T$, hence

$$
\frac{\partial \mathcal{L}}{\partial z_k} = \frac{1}{T}\left(q^{(T)}_k - p^{(T)}_k\right).
$$

*Step 3.* For large $T$ both tempered distributions flatten toward uniform. Expanding
$\operatorname{softmax}(z/T)_k = \tfrac1K + \tfrac{z_k - \bar z}{TK} + O(T^{-2})$ for
zero-mean logits gives
$q^{(T)}_k - p^{(T)}_k \approx (z_k - u_k)/(TK)$, so the full gradient scales like $T^{-2}$.

*Step 4.* Multiplying the distillation term by $T^2$ restores a gradient magnitude comparable to
the hard-label term, so the mixing weight does not have to be retuned whenever $T$ changes.

$$
\boxed{\nabla_z \mathcal{L} = \tfrac{1}{T}\left(q^{(T)} - p^{(T)}\right), \quad \text{multiply the loss by } T^2}
$$

**Key takeaway.** Temperature exposes the teacher's inter-class similarities while shrinking the
gradient; the $T^2$ factor cancels the shrinkage to first order.

In [19]:
u_teacher = np.array([3.0, 1.0, 0.5, -1.0])
z_student = np.array([2.0, 1.5, 0.0, -0.5])


def tempered(v, T):
    e = np.exp(v / T)
    return e / e.sum()


def loss_T(z, u, T):
    return float(-(tempered(u, T) * np.log(tempered(z, T))).sum())


print("   T     analytic (q-p)/T           finite difference          scaled-gradient norm * T^2")
for T in [1.0, 2.0, 4.0, 8.0]:
    ana = (tempered(z_student, T) - tempered(u_teacher, T)) / T
    fd = np.array([(loss_T(z_student + h, u_teacher, T) - loss_T(z_student - h, u_teacher, T)) / 2e-6
                   for h in np.eye(4) * 1e-6])
    print(f"{T:5.1f}   {np.round(ana, 6)}   {np.round(fd, 6)}   {T ** 2 * np.linalg.norm(ana):.6f}")
    assert np.abs(ana - fd).max() < 1e-7

   T     analytic (q-p)/T           finite difference          scaled-gradient norm * T^2
  1.0   [-0.261   0.223   0.0078  0.0302]   [-0.261   0.223   0.0078  0.0302]   0.344697
  2.0   [-0.0739  0.0573 -0.0044  0.0211]   [-0.0739  0.0573 -0.0044  0.0211]   0.383685
  4.0   [-0.017   0.0125 -0.0032  0.0076]   [-0.017   0.0125 -0.0032  0.0076]   0.362128
  8.0   [-0.0039  0.0028 -0.0011  0.0022]   [-0.0039  0.0028 -0.0011  0.0022]   0.343937


### Problem L2.5 — Auditing a reported loss against its floor

**Statement.** A $100$-class dataset has $20$ percent of its labels replaced uniformly by one of
the $99$ wrong classes. A paper reports a test cross-entropy of $0.30$ nats. Diagnose it.

**Intuition.** The best possible predictor still has to predict the *noise*, and its loss is the
entropy of the noisy label distribution.

**Solution.**

*Step 1.* Given the true class, the observed label is the true one with probability $0.8$ and
each wrong class with probability $0.2/99$.

*Step 2.* By Theorem 4.1 the optimal report is that distribution, so the floor is its entropy:

$$
H = -0.8 \ln 0.8 - 0.2 \ln \frac{0.2}{99} = 0.17851 + 0.2 \ln 495 = 1.41943 \text{ nats}.
$$

*Step 3.* The reported $0.30$ nats is far below the $1.41943$-nat floor. The floor is a bound on
any predictor whatsoever, so the report is impossible under the stated noise model.

*Step 4 — diagnoses.* The test labels are cleaner than claimed, the noise is not uniform or not
independent of the features, or — most commonly — the evaluation set overlaps the training set.

$$
\boxed{\text{floor} = 1.41943 \text{ nats} \gg 0.30 \text{ reported} \implies \text{leakage or a wrong noise model}}
$$

**Key takeaway.** Conditional-entropy floors turn loss numbers into audits: a loss below the
information-theoretic floor is a red flag, not an achievement.

In [20]:
Kc, noise = 100, 0.2
noisy = np.full(Kc, noise / (Kc - 1))
noisy[0] = 1 - noise
floor = entropy(noisy, np.e)
print("noisy label distribution: true class", noisy[0], " each wrong class", noisy[1])
print(f"floor H = {floor:.5f} nats")
print(f"  = -0.8 ln 0.8 + 0.2 ln 495 = {-0.8 * np.log(0.8):.5f} + {0.2 * np.log(495):.5f}")
print(f"reported 0.30 nats is below the floor by {floor - 0.30:.5f} nats")
assert abs(floor - 1.41943) < 1e-5
assert floor > 0.30

noisy label distribution: true class 0.8  each wrong class 0.00202020202020202
floor H = 1.41943 nats
  = -0.8 ln 0.8 + 0.2 ln 495 = 0.17851 + 1.24091
reported 0.30 nats is below the floor by 1.11943 nats


### Problem L2.6 — Class-weighted cross-entropy re-tilts the prior

**Statement.** A binary task has one positive per hundred examples. With weighted BCE
$\mathcal{L} = -w_+ y \ln \hat{p} - w_-(1-y)\ln(1-\hat{p})$ and $w_+ = 100$, $w_- = 1$, find the
optimal constant prediction and compare with the unweighted optimum.

**Intuition.** Weighting positives by $100$ makes the loss behave as if positives were $100$
times more common.

**Solution.**

*Step 1.* With $P(y=1) = 0.01$ the expected weighted loss of a constant $\hat{p}$ is

$$
\mathcal{L}(\hat{p}) = -0.01 \cdot 100 \ln \hat{p} - 0.99 \cdot 1 \cdot \ln(1 - \hat{p})
= -\ln \hat{p} - 0.99 \ln (1 - \hat{p}).
$$

*Step 2.* Stationarity gives

$$
\frac{1}{\hat{p}} = \frac{0.99}{1 - \hat{p}} \;\Longrightarrow\; \hat{p} = \frac{1}{1.99} = 0.50251 .
$$

*Step 3.* The unweighted optimum is the base rate $0.01$ by Problem L1.2.

*Step 4.* The weighted loss is no longer proper for the original distribution; it is proper for
the reweighted one, in which positives have probability $w_+ p / (w_+ p + w_-(1-p))$.

$$
\boxed{\hat{p}^{\star}_{\text{weighted}} = \tfrac{1}{1.99} = 0.50251 \quad\text{against}\quad \hat{p}^{\star}_{\text{unweighted}} = 0.01}
$$

**Key takeaway.** Class weights buy balanced gradients at the cost of calibration: the outputs
are honest probabilities for the reweighted distribution, and must be corrected before being
read as probabilities of the original distribution.

In [21]:
base, wp, wm = 0.01, 100.0, 1.0
grid26 = np.linspace(1e-5, 1 - 1e-5, 200001)
w_loss = -base * wp * np.log(grid26) - (1 - base) * wm * np.log(1 - grid26)
u_loss = -base * np.log(grid26) - (1 - base) * np.log(1 - grid26)
print(f"weighted   argmin = {grid26[w_loss.argmin()]:.5f}   predicted {1 / 1.99:.5f}")
print(f"unweighted argmin = {grid26[u_loss.argmin()]:.5f}   predicted {base:.5f}")
tilted = wp * base / (wp * base + wm * (1 - base))
print(f"reweighted positive rate = {tilted:.5f}")
assert abs(grid26[w_loss.argmin()] - 1 / 1.99) < 1e-4
assert abs(grid26[u_loss.argmin()] - base) < 1e-4
assert abs(tilted - 1 / 1.99) < 1e-12

weighted   argmin = 0.50251   predicted 0.50251
unweighted argmin = 0.01000   predicted 0.01000
reweighted positive rate = 0.50251


### Problem L2.7 — Landauer erasure with a mismatched code

**Statement.** A molecular memory holds $N = 10^{23}$ symbols drawn i.i.d. from
$p = (\tfrac12, \tfrac14, \tfrac18, \tfrac18)$. Landauer's principle says erasing one bit of
information at temperature $T$ dissipates at least $k_B T \ln 2$ joules. The memory is
compressed with a Shannon code before erasure. Compute the minimum heat when the code is matched
to $p$, when it is matched to the uniform $q$, and the excess, at $T = 300$ K.

**Intuition.** Erasure cost is proportional to the number of bits actually stored, and a
mismatched code stores more bits than necessary.

**Solution.**

*Step 1 — cost of one bit.* With $k_B = 1.380649 \times 10^{-23}$ J/K,

$$
k_B T \ln 2 = 1.380649 \times 10^{-23} \times 300 \times 0.693147 = 2.87098 \times 10^{-21} \text{ J}.
$$

*Step 2 — matched code.* By Theorem 4.3 with $q = p$ dyadic, the code stores exactly
$H(p) = 1.75$ bits per symbol, so

$$
Q_{\min} = 1.75 \times 10^{23} \times 2.87098 \times 10^{-21} = 502.42 \text{ J}.
$$

*Step 3 — mismatched code.* The uniform code stores $H_{\times}(p, q) = 2$ bits per symbol:

$$
Q = 2 \times 10^{23} \times 2.87098 \times 10^{-21} = 574.20 \text{ J}.
$$

*Step 4 — excess.* The difference is
$D_{\mathrm{KL}}(p \parallel q) = 0.25$ bits per symbol, so

$$
\Delta Q = 0.25 \times 10^{23} \times 2.87098 \times 10^{-21} = 71.77 \text{ J}.
$$

$$
\boxed{Q_{\min} = 502.42 \text{ J}, \quad Q_{\text{uniform}} = 574.20 \text{ J}, \quad \Delta Q = k_B T \ln 2 \cdot N \cdot D_{\mathrm{KL}} = 71.77 \text{ J}}
$$

**Key takeaway.** The KL divergence of a modelling error is a physical quantity: at room
temperature, believing the wrong distribution about $10^{23}$ symbols costs an extra $72$ joules
of waste heat.

In [22]:
kB, T_room = 1.380649e-23, 300.0
e_bit = kB * T_room * np.log(2)
p27 = np.array([0.5, 0.25, 0.125, 0.125])
u27 = np.full(4, 0.25)
N27 = 1e23
print(f"k_B T ln2 = {e_bit:.5e} J per bit at {T_room} K")
print(f"H(p)          = {entropy(p27):.4f} bits/symbol -> Q = {entropy(p27) * N27 * e_bit:.2f} J")
print(f"H_x(p, u)     = {cross_entropy(p27, u27):.4f} bits/symbol -> Q = {cross_entropy(p27, u27) * N27 * e_bit:.2f} J")
print(f"D_KL(p || u)  = {kl(p27, u27):.4f} bits/symbol -> excess = {kl(p27, u27) * N27 * e_bit:.2f} J")
assert abs(entropy(p27) * N27 * e_bit - 502.42) < 0.01
assert abs(cross_entropy(p27, u27) * N27 * e_bit - 574.20) < 0.01
assert abs(kl(p27, u27) * N27 * e_bit - 71.77) < 0.01

k_B T ln2 = 2.87098e-21 J per bit at 300.0 K
H(p)          = 1.7500 bits/symbol -> Q = 502.42 J
H_x(p, u)     = 2.0000 bits/symbol -> Q = 574.20 J
D_KL(p || u)  = 0.2500 bits/symbol -> excess = 71.77 J


### Problem L2.8 — Free energy of a two-level system at the wrong temperature

**Statement.** A two-level system has energies $E_0 = 0$ and $E_1 = \Delta = 0.02$ eV and sits at
$T = 300$ K. Show that the Gibbs free energy of a trial distribution $q$ satisfies
$F(q) = F^{\star} + k_B T \, D_{\mathrm{KL}}(q \parallel p^{\star})$, then evaluate the penalty
when $q$ is the Boltzmann distribution for $T' = 400$ K.

**Intuition.** Mean energy is a cross-entropy against the Boltzmann distribution, so the free
energy excess is a KL divergence.

**Solution.**

*Step 1 — rewrite the energies.* With $p^{\star}(x) = e^{-E_x / k_B T}/Z$ we get
$E_x = -k_B T \ln\left(Z p^{\star}(x)\right)$, hence, with entropy in nats,

$$
\mathbb{E}_q[E] = k_B T \, H_{\times}(q, p^{\star}) - k_B T \ln Z .
$$

*Step 2 — assemble the free energy.*

$$
F(q) = \mathbb{E}_q[E] - k_B T H(q) = k_B T \left(H_{\times}(q, p^{\star}) - H(q)\right) - k_B T \ln Z
= -k_B T \ln Z + k_B T \, D_{\mathrm{KL}}(q \parallel p^{\star}),
$$

using Theorem 4.1. Since $D_{\mathrm{KL}} \ge 0$, equilibrium minimizes $F$ and
$F^{\star} = -k_B T \ln Z$.

*Step 3 — the numbers.* At $T = 300$ K, $k_B T = 4.14195 \times 10^{-21}$ J and
$\Delta/k_B T = 0.77363$, so

$$
p^{\star} = (0.68431, 0.31569), \qquad Z = 1.46133, \qquad F^{\star} = -0.00981 \text{ eV}.
$$

At $T' = 400$ K, $\Delta / k_B T' = 0.58023$ and $q = (0.64112, 0.35888)$.

*Step 4 — the penalty.*

$$
D_{\mathrm{KL}}(q \parallel p^{\star}) = 0.0042202 \text{ nats},
\qquad
F(q) - F^{\star} = k_B T \, D_{\mathrm{KL}} = 1.74800 \times 10^{-23} \text{ J} = 1.09102 \times 10^{-4} \text{ eV}.
$$

$$
\boxed{F(q) - F^{\star} = k_B T \, D_{\mathrm{KL}}(q \parallel p^{\star}) = 1.09102 \times 10^{-4} \text{ eV} = 0.00422\, k_B T}
$$

**Key takeaway.** The second law and Gibbs' inequality are the same statement: any trial
distribution costs $k_B T$ times its divergence from equilibrium, which is exactly the ELBO gap
in variational inference.

In [23]:
eV = 1.602176634e-19
Delta = 0.02 * eV
kT, kT2 = kB * 300.0, kB * 400.0
p_star = np.array([1.0, np.exp(-Delta / kT)])
p_star /= p_star.sum()
q_wrong = np.array([1.0, np.exp(-Delta / kT2)])
q_wrong /= q_wrong.sum()
Z = 1 + np.exp(-Delta / kT)
F_star = -kT * np.log(Z)
F_q = q_wrong[1] * Delta - kT * entropy(q_wrong, np.e)
D = kl(q_wrong, p_star, np.e)
print(f"Delta/kT  = {Delta / kT:.5f}   Delta/kT' = {Delta / kT2:.5f}")
print("p* =", p_star, "  q =", q_wrong)
print(f"Z = {Z:.5f}   F* = {F_star:.5e} J = {F_star / eV:.5f} eV")
print(f"F(q) - F* = {F_q - F_star:.5e} J     kT * KL = {kT * D:.5e} J")
print(f"KL(q || p*) = {D:.7f} nats   penalty = {(F_q - F_star) / eV:.5e} eV = {D:.5f} kT")
assert abs((F_q - F_star) - kT * D) < 1e-30
assert abs(D - 0.0042202) < 1e-7

Delta/kT  = 0.77363   Delta/kT' = 0.58023
p* = [0.6843 0.3157]   q = [0.6411 0.3589]
Z = 1.46133   F* = -1.57124e-21 J = -0.00981 eV
F(q) - F* = 1.74800e-23 J     kT * KL = 1.74800e-23 J
KL(q || p*) = 0.0042202 nats   penalty = 1.09102e-04 eV = 0.00422 kT


## L3 — Challenge Proofs

### Problem L3.1 — The Brier score is proper but not local

**Statement.** The Brier score for a report $q$ on a $K$-outcome event is
$S(q, x) = \sum_k \left(q_k - \mathbf{1}\{x = k\}\right)^2$. Prove that it is strictly proper,
and show it is not local.

**Intuition.** Expanding the square turns the expected score into a squared distance plus a term
that does not depend on the report.

**Solution.**

*Step 1 — expected score.* Using $\mathbb{E}\left[\mathbf{1}\{X=k\}\right] = p_k$ and
$\mathbf{1}^2 = \mathbf{1}$,

$$
\bar{S}(p, q) = \sum_k \left(q_k^2 - 2 q_k p_k + p_k\right).
$$

*Step 2 — subtract the honest score.*

$$
\bar{S}(p, q) - \bar{S}(p, p) = \sum_k \left(q_k^2 - 2q_k p_k + p_k^2\right) = \sum_k (q_k - p_k)^2 \ge 0,
$$

with equality if and only if $q = p$: strictly proper.

*Step 3 — non-locality.* $S(q, x)$ contains every $q_k^2$, so two reports agreeing on $q_x$ but
differing elsewhere receive different scores when $x$ occurs. Example 6.5 exhibits the pair
$(0.4, 0.4, 0.2)$ and $(0.4, 0.5, 0.1)$, scoring $0.56$ against $0.62$ on outcome $1$ while the
log score gives both $-\ln 0.4$.

*Step 4 — consequence.* By Theorem 4.7 no smooth strictly proper local rule other than an affine
transform of $-\log q_x$ exists once $K \ge 3$, so Brier's non-locality is not a defect of this
particular rule but the price of leaving the logarithm.

$$
\boxed{\bar{S}(p,q) - \bar{S}(p,p) = \sum_k (q_k - p_k)^2; \text{ proper, but not local}}
$$

**Key takeaway.** Properness admits many rules; locality singles out the logarithm, and with it
the whole apparatus of entropy and code length.

In [24]:
p31 = np.array([0.5, 0.3, 0.2])


def brier(p, r):
    return float(sum(p[k] * ((r - np.eye(len(p))[k]) ** 2).sum() for k in range(len(p))))


for _ in range(4):
    r = rng.dirichlet(np.ones(3))
    print(f"report {np.round(r, 4)}:  gap = {brier(p31, r) - brier(p31, p31):.6f}"
          f"   ||r - p||^2 = {((r - p31) ** 2).sum():.6f}")
    assert abs((brier(p31, r) - brier(p31, p31)) - ((r - p31) ** 2).sum()) < 1e-12
qa, qb = np.array([0.4, 0.4, 0.2]), np.array([0.4, 0.5, 0.1])
e1 = np.array([1.0, 0.0, 0.0])
print("\noutcome 1 occurs:")
print(f"  Brier(q) = {((qa - e1) ** 2).sum():.4f}   Brier(r) = {((qb - e1) ** 2).sum():.4f}")
print(f"  log score both = {-np.log(qa[0]):.5f}")
assert abs(((qa - e1) ** 2).sum() - ((qb - e1) ** 2).sum()) > 0.05

report [0.001  0.2522 0.7468]:  gap = 0.550246   ||r - p||^2 = 0.550246
report [0.1587 0.1779 0.6634]:  gap = 0.346212   ||r - p||^2 = 0.346212
report [0.6482 0.3517 0.0001]:  gap = 0.064578   ||r - p||^2 = 0.064578
report [0.6652 0.0213 0.3135]:  gap = 0.117886   ||r - p||^2 = 0.117886

outcome 1 occurs:
  Brier(q) = 0.5600   Brier(r) = 0.6200
  log score both = 0.91629


### Problem L3.2 — Optimal code lengths under the Kraft constraint

**Statement.** Real-valued codeword lengths $\ell_x$ must satisfy Kraft's inequality
$\sum_x 2^{-\ell_x} \le 1$ (Theorem 4.2). Prove that $\sum_x p(x)\ell_x$ is minimized at
$\ell^{\star}_x = -\log_2 p(x)$ with minimum $H(p)$, and interpret the excess for any other
choice.

**Intuition.** Kraft-tight length assignments are exactly distributions in disguise.

**Solution.**

*Step 1.* At an optimum Kraft holds with equality: if $\sum_x 2^{-\ell_x} \lt 1$, decreasing any
$\ell_x$ slightly keeps feasibility and lowers the objective.

*Step 2.* Define $q(x) = 2^{-\ell_x}$. Kraft equality says $q$ is a pmf, and
$\ell_x = -\log_2 q(x)$.

*Step 3.* The objective becomes a cross-entropy:

$$
\sum_x p(x) \ell_x = -\sum_x p(x) \log_2 q(x) = H_{\times}(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q).
$$

*Step 4.* By Theorem 4.1 the minimum over $q$ is at $q = p$, so
$\ell^{\star}_x = -\log_2 p(x)$ and the minimum is $H(p)$. Any other assignment corresponds to
some $q \neq p$ and pays exactly $D_{\mathrm{KL}}(p \parallel q)$ extra bits per symbol; rounding
to integers costs strictly less than one further bit by Theorem 4.3.

$$
\boxed{\ell^{\star}(x) = -\log_2 p(x), \qquad \min \mathbb{E}[\ell] = H(p), \qquad \text{excess} = D_{\mathrm{KL}}(p \parallel q)}
$$

**Key takeaway.** "Cross-entropy is the expected length of the wrong code" is a change of
variables, $q(x) = 2^{-\ell(x)}$, not an analogy.

In [25]:
p32 = np.array([0.45, 0.3, 0.15, 0.1])
ell_star = -np.log2(p32)
print("optimal real lengths:", np.round(ell_star, 4), " Kraft =", (2.0 ** -ell_star).sum())
print(f"E[l*] = {float(p32 @ ell_star):.5f} bits = H(p) = {entropy(p32):.5f} bits")
best = float(p32 @ ell_star)
for _ in range(4):
    q = rng.dirichlet(np.ones(4))
    ell = -np.log2(q)
    print(f"  q = {np.round(q, 3)}: E[l] = {float(p32 @ ell):.5f}"
          f"   excess = {float(p32 @ ell) - best:.5f}   KL = {kl(p32, q):.5f}")
    assert abs((float(p32 @ ell) - best) - kl(p32, q)) < 1e-12
ell_int = np.ceil(ell_star)
print(f"\ninteger (Shannon) lengths {ell_int.astype(int)}: E[l] = {float(p32 @ ell_int):.5f} bits"
      f"   overhead = {float(p32 @ ell_int) - best:.5f} bits (< 1)")
assert 0 <= float(p32 @ ell_int) - best < 1

optimal real lengths: [1.152  1.737  2.737  3.3219]  Kraft = 1.0
E[l*] = 1.78223 bits = H(p) = 1.78223 bits
  q = [0.182 0.676 0.076 0.066]: E[l] = 2.22509   excess = 0.44286   KL = 0.44286
  q = [0.554 0.014 0.05  0.382]: E[l] = 3.02567   excess = 1.24344   KL = 1.24344
  q = [0.23  0.515 0.123 0.133]: E[l] = 1.98801   excess = 0.20578   KL = 0.20578
  q = [0.45  0.42  0.079 0.051]: E[l] = 1.87240   excess = 0.09018   KL = 0.09018

integer (Shannon) lengths [2 2 3 4]: E[l] = 2.35000 bits   overhead = 0.56777 bits (< 1)


### Problem L3.3 — Kelly betting and the doubling rate

**Statement.** A gambler bets fractions $b_x$ of wealth on a $K$-outcome race with true
probabilities $p$ and payoff odds $o_x$. Prove that the asymptotic growth rate is
$W(b) = \sum_x p(x)\log(o_x b_x)$, specialize to fair odds $o_x = K$ to get
$W(b) = \log K - H_{\times}(p, b)$, and show that using beliefs $q \neq p$ costs exactly
$D_{\mathrm{KL}}(p \parallel q)$ per race.

**Intuition.** Wealth multiplies, so its logarithm adds, and the law of large numbers turns the
sum into an expectation.

**Solution.**

*Step 1.* After a race with outcome $x$, wealth multiplies by $o_x b_x$. After $n$ i.i.d. races,

$$
\frac{1}{n}\log \frac{V_n}{V_0} = \frac{1}{n}\sum_{i=1}^{n} \log\left(o_{X_i} b_{X_i}\right)
\xrightarrow[\text{LLN}]{} \mathbb{E}_p\left[\log\left(o_X b_X\right)\right] = W(b) .
$$

*Step 2 — fair odds.* Setting $o_x = K$ for every $x$,

$$
W(b) = \log K + \sum_x p(x)\log b_x = \log K - H_{\times}(p, b) .
$$

The additive $\log K$ is what makes $W$ positive for a good bettor; without it the growth rate
of every strategy would be negative.

*Step 3 — the optimum.* Maximizing $W$ is minimizing $H_{\times}(p, \cdot)$, so by Theorem 4.1
the optimum is $b^{\star} = p$ with $W^{\star} = \log K - H(p)$.

*Step 4 — the cost of wrong beliefs.*

$$
W(p) - W(q) = H_{\times}(p, q) - H(p) = D_{\mathrm{KL}}(p \parallel q) .
$$

With a track take, $o_x = (1-t)K$ and $W$ drops by $\log \frac{1}{1-t}$ for every strategy, so a
large enough take makes even the Kelly bettor lose.

$$
\boxed{W(b) = \log K - H_{\times}(p, b), \qquad b^{\star} = p, \qquad \text{cost of } q = D_{\mathrm{KL}}(p \parallel q)}
$$

**Key takeaway.** Entropy is the house's structural take and KL is the price of wrong beliefs,
both measured in doubling rate — log-loss evaluation is a betting market.

In [26]:
p33 = np.array([0.5, 0.25, 0.15, 0.10])
K33 = p33.size
q33 = np.array([0.4, 0.3, 0.2, 0.1])
races = 20000
draws = rng.choice(K33, size=races, p=p33)
for name, b in [("b = p", p33), ("b = q", q33), ("b = uniform", np.full(K33, 1 / K33))]:
    pred = np.log2(K33) - cross_entropy(p33, b)
    meas = float(np.log2(K33 * b[draws]).mean())
    print(f"{name:12s} predicted W = {pred:+.5f}   measured = {meas:+.5f} bits/race")
print(f"\ncost of betting q       = {kl(p33, q33):.5f} bits/race = D_KL(p || q)")
print(f"W(p) - W(q)             = {(np.log2(K33) - cross_entropy(p33, p33)) - (np.log2(K33) - cross_entropy(p33, q33)):.5f}")
take = 0.15
print(f"with a {take:.0%} take, o = {(1 - take) * K33:.2f}:  W(p) = "
      f"{np.log2((1 - take) * K33) - entropy(p33):+.5f} bits/race")
assert abs(((np.log2(K33) - cross_entropy(p33, p33)) - (np.log2(K33) - cross_entropy(p33, q33)))
           - kl(p33, q33)) < 8 * EPS

b = p        predicted W = +0.25726   measured = +0.25360 bits/race
b = q        predicted W = +0.22431   measured = +0.22371 bits/race
b = uniform  predicted W = +0.00000   measured = +0.00000 bits/race

cost of betting q       = 0.03295 bits/race = D_KL(p || q)
W(p) - W(q)             = 0.03295
with a 15% take, o = 3.40:  W(p) = +0.02280 bits/race


### Problem L3.4 — Calibration and refinement decomposition of log loss

**Statement.** A forecaster emits predictions $Q$ for a binary event $Y$, where $Q$ takes
**finitely many** values, and let $c(q) = P(Y = 1 \mid Q = q)$. Prove

$$
\mathbb{E}\left[\mathrm{BCE}\right]
= \underbrace{\mathbb{E}_Q\left[D_{\mathrm{KL}}\left(\mathrm{Ber}(c(Q)) \parallel \mathrm{Ber}(Q)\right)\right]}_{\text{calibration}}
+ \underbrace{\mathbb{E}_Q\left[H_b\left(c(Q)\right)\right]}_{\text{refinement}} ,
$$

and interpret both terms.

**Intuition.** Condition on what the forecaster said; given that, the event is a coin whose bias
is the calibration curve.

**Solution.**

*Step 1 — condition on the report.* The finite-alphabet hypothesis makes $c(q)$ well defined for
every emitted value, with no conditioning subtleties. Given $Q = q$, $Y$ is Bernoulli with
parameter $c(q)$ and the loss scores the constant report $q$:

$$
\mathbb{E}\left[\mathrm{BCE} \mid Q = q\right] = -c(q)\ln q - \left(1 - c(q)\right)\ln(1 - q)
= H_{\times}\left(\mathrm{Ber}(c(q)), \mathrm{Ber}(q)\right).
$$

*Step 2 — decompose the Bernoulli cross-entropy.* By Theorem 4.1,

$$
H_{\times}\left(\mathrm{Ber}(c(q)), \mathrm{Ber}(q)\right)
= H_b\left(c(q)\right) + D_{\mathrm{KL}}\left(\mathrm{Ber}(c(q)) \parallel \mathrm{Ber}(q)\right).
$$

*Step 3 — average over the report.* Taking the expectation over $Q$ and using the tower property
gives the claim.

*Step 4 — interpretation.* The calibration term vanishes exactly when $c(q) = q$ for every
emitted $q$ — predictions mean what they say — and temperature scaling attacks only this term.
The refinement term is the residual uncertainty after conditioning on the forecast; it is small
when forecasts separate the classes sharply, and it is bounded below by $H(Y \mid X)$ for any
forecaster that is a function of features $X$.

*Remark.* For a continuous forecast distribution $c$ is defined only up to a null set, and in
practice it is estimated by binning — which is exactly what a reliability diagram plots.

$$
\boxed{\mathbb{E}\left[\mathrm{BCE}\right] = \text{calibration KL} + \text{refinement entropy}}
$$

**Key takeaway.** Log loss audits two virtues at once, honesty and sharpness, and this
decomposition separates the ledger.

In [27]:
report_values = np.array([0.1, 0.35, 0.6, 0.9])
report_probs = np.array([0.4, 0.25, 0.2, 0.15])
calib = np.array([0.15, 0.30, 0.65, 0.80])
n34 = 400000
idx = rng.choice(4, size=n34, p=report_probs)
Qv, Cv = report_values[idx], calib[idx]
Yv = (rng.random(n34) < Cv).astype(float)
empirical = float(np.mean(-Yv * np.log(Qv) - (1 - Yv) * np.log(1 - Qv)))
cal_term = float(sum(report_probs[j] * kl(np.array([calib[j], 1 - calib[j]]),
                                          np.array([report_values[j], 1 - report_values[j]]), np.e)
                     for j in range(4)))
ref_term = float(sum(report_probs[j] * binary_entropy(calib[j], np.e) for j in range(4)))
print(f"empirical mean BCE     = {empirical:.5f} nats")
print(f"calibration term       = {cal_term:.5f} nats")
print(f"refinement term        = {ref_term:.5f} nats")
print(f"calibration + refinement = {cal_term + ref_term:.5f} nats")
print(f"perfectly calibrated forecaster (c = q) would score {ref_term:.5f} + 0")
assert abs(empirical - (cal_term + ref_term)) < 5e-3

empirical mean BCE     = 0.53920 nats
calibration term       = 0.01402 nats
refinement term        = 0.52635 nats
calibration + refinement = 0.54037 nats
perfectly calibrated forecaster (c = q) would score 0.52635 + 0


### Problem L3.5 — Savage representation: every proper score is a Bregman divergence

**Statement.** Let $S$ be a proper scoring rule and put
$G(p) = \bar{S}(p, p) = \min_q \bar{S}(p, q)$, assumed differentiable on the interior of the
simplex. Prove

$$
\bar{S}(p, q) - \bar{S}(p, p) = -G(p) + G(q) + \left\langle \nabla G(q),\, p - q \right\rangle,
$$

the Bregman divergence generated by the convex function $-G$, and identify $G$ for the log
score.

**Intuition.** For fixed $q$ the expected score is affine in $p$, sits above $G$, and touches it
at $p = q$ — so it is the tangent plane.

**Solution.**

*Step 1 — $G$ is concave.* $\bar{S}(\cdot, q)$ is affine in $p$ for each fixed $q$, and $G$ is
the pointwise minimum of that family, so $G$ is concave.

*Step 2 — the affine function touches.* Properness gives
$\bar{S}(p, q) \ge \bar{S}(p, p) = G(p)$ for all $p$, with equality at $p = q$. So
$p \mapsto \bar{S}(p, q)$ is an affine function lying above $G$ and touching it at $q$.

*Step 3 — identify it with the tangent.* A differentiable concave $G$ has exactly one affine
majorant touching at $q$, namely $p \mapsto G(q) + \langle \nabla G(q), p - q\rangle$. Hence
$\bar{S}(p, q) = G(q) + \langle \nabla G(q), p - q\rangle$.

*Step 4 — subtract.*

$$
\bar{S}(p, q) - G(p) = G(q) + \left\langle \nabla G(q), p - q\right\rangle - G(p) = D_{-G}(p, q) \ge 0,
$$

the Bregman divergence of the convex $-G$, non-negative by convexity.

*Step 5 — the log score.* Here $\bar{S}(p,q) = H_{\times}(p,q)$ and $G(p) = H(p)$, so $-G$ is
the negative entropy and the Bregman divergence it generates is $D_{\mathrm{KL}}$ — recovering
Theorem 4.1.

$$
\boxed{\bar{S}(p, q) - \bar{S}(p, p) = D_{-G}(p, q), \qquad G = H \Rightarrow D_{-G} = D_{\mathrm{KL}}}
$$

**Key takeaway.** Proper scoring rules and Bregman divergences are the same objects seen from
two sides; log loss is the one whose generator is the entropy, and Brier the one whose generator
is $\lVert p \rVert_2^2$.

In [28]:
def bregman(p, q, grad, gen):
    return float(gen(p) - gen(q) - grad(q) @ (p - q))


neg_entropy = lambda v: float((v * np.log(v)).sum())
grad_neg_entropy = lambda v: np.log(v) + 1.0
sq_norm = lambda v: float((v ** 2).sum())
grad_sq_norm = lambda v: 2.0 * v

print("generator = negative entropy  -> Bregman should equal KL (nats)")
for _ in range(3):
    a, b = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3))
    print(f"  Bregman = {bregman(a, b, grad_neg_entropy, neg_entropy):.8f}"
          f"   KL = {kl(a, b, np.e):.8f}")
    assert abs(bregman(a, b, grad_neg_entropy, neg_entropy) - kl(a, b, np.e)) < 1e-10

print("\ngenerator = squared norm      -> Bregman should equal ||p - q||^2")
for _ in range(3):
    a, b = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3))
    print(f"  Bregman = {bregman(a, b, grad_sq_norm, sq_norm):.8f}"
          f"   ||a-b||^2 = {((a - b) ** 2).sum():.8f}")
    assert abs(bregman(a, b, grad_sq_norm, sq_norm) - ((a - b) ** 2).sum()) < 1e-10

generator = negative entropy  -> Bregman should equal KL (nats)
  Bregman = 0.23673202   KL = 0.23673202
  Bregman = 0.27326784   KL = 0.27326784
  Bregman = 0.01613162   KL = 0.01613162

generator = squared norm      -> Bregman should equal ||p - q||^2
  Bregman = 1.08816788   ||a-b||^2 = 1.08816788
  Bregman = 0.10460824   ||a-b||^2 = 0.10460824
  Bregman = 0.79294785   ||a-b||^2 = 0.79294785


### Problem L3.6 — The Fisher-information expansion of the cross-entropy gap

**Statement.** Let $\{p_\theta\}$ be a smooth family of pmfs with Fisher information
$I(\theta) = \mathbb{E}_{p_\theta}\left[\left(\partial_\theta \ln p_\theta(X)\right)^2\right]$.
Prove

$$
H_{\times}\left(p_\theta, p_{\theta + \delta}\right) - H\left(p_\theta\right)
= D_{\mathrm{KL}}\left(p_\theta \parallel p_{\theta+\delta}\right)
= \tfrac12 I(\theta)\, \delta^2 + O\!\left(\delta^3\right),
$$

and verify it on the Bernoulli family at $\theta = 0.3$.

**Intuition.** The KL divergence has a minimum of zero at $\delta = 0$, so its leading behaviour
is quadratic, and the curvature is the Fisher information.

**Solution.**

*Step 1 — the gap is a KL.* Immediate from Theorem 4.1 with $p = p_\theta$ and
$q = p_{\theta+\delta}$.

*Step 2 — expand in $\delta$.* Write $f(\delta) = D_{\mathrm{KL}}(p_\theta \parallel p_{\theta+\delta})$.
Then $f(0) = 0$, and

$$
f'(\delta) = -\sum_x p_\theta(x)\, \frac{\partial_\theta p_{\theta+\delta}(x)}{p_{\theta+\delta}(x)},
\qquad
f'(0) = -\sum_x \partial_\theta p_\theta(x) = -\partial_\theta \sum_x p_\theta(x) = 0 ,
$$

because probabilities sum to the constant $1$.

*Step 3 — the second derivative.* Differentiating again and evaluating at $\delta = 0$,

$$
f''(0) = \sum_x p_\theta(x)\left(\frac{\partial_\theta p_\theta(x)}{p_\theta(x)}\right)^2
- \sum_x \partial^2_\theta p_\theta(x) = I(\theta) - 0 = I(\theta),
$$

the second sum vanishing for the same reason as in Step 2.

*Step 4 — Taylor.* Hence $f(\delta) = \tfrac12 I(\theta)\delta^2 + O(\delta^3)$.

*Step 5 — Bernoulli check.* For $p_\theta = (\theta, 1-\theta)$,
$I(\theta) = 1/\left(\theta(1-\theta)\right)$, which at $\theta = 0.3$ is $4.76190$. So the gap
at $\delta = 0.01$ should be about $\tfrac12 (4.76190)(10^{-4}) = 2.38095 \times 10^{-4}$ nats,
against an exact value of $2.35169 \times 10^{-4}$.

$$
\boxed{D_{\mathrm{KL}}\left(p_\theta \parallel p_{\theta+\delta}\right) = \tfrac12 I(\theta)\delta^2 + O(\delta^3)}
$$

**Key takeaway.** Near the optimum cross-entropy is a quadratic form with the Fisher information
as its Hessian, which is why natural gradient and Newton steps agree there and why
Proposition 4.10 gives $(K-1)/(2N)$ — one half per free parameter.

In [29]:
theta = 0.3
I_theta = 1.0 / (theta * (1 - theta))


def kl_bernoulli(a, b):
    return float(a * np.log(a / b) + (1 - a) * np.log((1 - a) / (1 - b)))


print(f"I(theta) = {I_theta:.5f} at theta = {theta}")
print("   delta      exact KL        quadratic        ratio")
prev = None
for d in [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003]:
    exact, quad = kl_bernoulli(theta, theta + d), 0.5 * I_theta * d * d
    print(f"  {d:8.4f}   {exact:.6e}   {quad:.6e}   {exact / quad:.6f}")
    prev = exact / quad
assert abs(prev - 1.0) < 1e-3
assert abs(kl_bernoulli(0.3, 0.31) - 2.35169e-4) < 1e-8

I(theta) = 4.76190 at theta = 0.3
   delta      exact KL        quadratic        ratio
    0.1000   2.160085e-02   2.380952e-02   0.907236
    0.0300   2.068782e-03   2.142857e-03   0.965432
    0.0100   2.351694e-04   2.380952e-04   0.987711
    0.0030   2.134774e-05   2.142857e-05   0.996228
    0.0010   2.377939e-06   2.380952e-06   0.998734
    0.0003   2.142042e-07   2.142857e-07   0.999619
